In [11]:
import numpy as np

def parse_sign_matrix(text):
    """conver + to +1, - to -1"""
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    return np.array([[1 if c == '+' else -1 for c in line] for line in lines], dtype=int)

def normalize_pm_matrix(H):
    """normalize ±1 matrix s.t. first row and first column are all +1"""
    H = H.copy()
    n = len(H)
    for j in range(n):
        if H[0, j] == -1: H[:, j] *= -1
    for i in range(n):
        if H[i, 0] == -1: H[i, :] *= -1
    return H

def reduce_to_01_matrix(text):
    """conver NxN ±1 matrix to (N-1)x(N-1) 0/1 matrix"""
    H = normalize_pm_matrix(parse_sign_matrix(text))
    B = H[1:, 1:]
    A = (1 - B) // 2   # +1→0, -1→1
    return A


text = """
+-------------------
++-++----+-+-++++--+
+++-++----+-+-++++--
+-++-++----+-+-++++-
+--++-++----+-+-++++
++--++-++----+-+-+++
+++--++-++----+-+-++
++++--++-++----+-+-+
+++++--++-++----+-+-
+-++++--++-++----+-+
++-++++--++-++----+-
+-+-++++--++-++----+
++-+-++++--++-++----
+-+-+-++++--++-++---
+--+-+-++++--++-++--
+---+-+-++++--++-++-
+----+-+-++++--++-++
++----+-+-++++--++-+
+++----+-+-++++--++-
+-++----+-+-++++--++
"""
A = reduce_to_01_matrix(text)
print("0/1 matrix:\n", A)
# det of A
print(abs(np.linalg.det(A)))

formatted = "upper_bound_matrix = [\n    " + ";\n    ".join(
    [" ".join(map(str, row)) for row in A]
) + "\n];"

print(formatted)

0/1 matrix:
 [[1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 0 0 1]
 [1 1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 0 0]
 [0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 0]
 [0 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1]
 [1 0 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 1 1]
 [1 1 0 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 1]
 [1 1 1 0 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1]
 [1 1 1 1 0 0 1 1 0 1 1 0 0 0 0 1 0 1 0]
 [0 1 1 1 1 0 0 1 1 0 1 1 0 0 0 0 1 0 1]
 [1 0 1 1 1 1 0 0 1 1 0 1 1 0 0 0 0 1 0]
 [0 1 0 1 1 1 1 0 0 1 1 0 1 1 0 0 0 0 1]
 [1 0 1 0 1 1 1 1 0 0 1 1 0 1 1 0 0 0 0]
 [0 1 0 1 0 1 1 1 1 0 0 1 1 0 1 1 0 0 0]
 [0 0 1 0 1 0 1 1 1 1 0 0 1 1 0 1 1 0 0]
 [0 0 0 1 0 1 0 1 1 1 1 0 0 1 1 0 1 1 0]
 [0 0 0 0 1 0 1 0 1 1 1 1 0 0 1 1 0 1 1]
 [1 0 0 0 0 1 0 1 0 1 1 1 1 0 0 1 1 0 1]
 [1 1 0 0 0 0 1 0 1 0 1 1 1 1 0 0 1 1 0]
 [0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 0 0 1 1]]
19531249.99999996
upper_bound_matrix = [
    1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 0 0 1;
    1 1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 0 0;
    0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 0;
    0 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 1

In [ ]:
# Modified version to store matrices for each determinant value
from collections import Counter, defaultdict

N = 17

def decode_matrix(line):
    """decode a line of 0/1 string to a matrix"""
    try: 
        row_strs = line.strip().split(",")

        if len(row_strs) != N:
            return None
        
        matrix = []
        for row in row_strs:
            # validate row
            if len(row) != N:
                return None
            if not all(c in "01" for c in row):
                return None

            matrix.append([int(c) for c in row])

        matrix = np.array(matrix, dtype=int)

        if matrix.shape != (N, N):
            return None  
        return matrix
    
    except Exception:
        return None

def canonical_form(A):
    """return the canonical form of matrix A under row and column permutations"""
    A_sorted_rows = np.array(sorted(A.tolist()))
    A_sorted_cols = np.array(sorted(A_sorted_rows.T.tolist())).T
    return ''.join(map(str, A_sorted_cols.flatten().tolist()))

file_path = "C:/Users/123li/Downloads/Project/gpu_run_output/dim17_run_5/1111/transformer-output-decoded.txt"
with open(file_path, encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

invalid_count = 0
canonical_strings = {}
det_to_matrices = defaultdict(list)  # Store matrices for each determinant

for line in lines:
    A = decode_matrix(line)
    if A is None:
        invalid_count += 1
        continue
    
    can_str = canonical_form(A)
    if can_str not in canonical_strings:  # ensure no repeats
        det = round(abs(np.linalg.det(A))) 
        canonical_strings[can_str] = det
        # Store the matrix for this determinant
        det_to_matrices[det].append(A.copy())

print(f"Total matrices: {len(lines)}")
print(f"Unique (up to permutation): {len(canonical_strings)}")
print(f"Duplicate rate: {(1 - len(canonical_strings)/len(lines))*100:.2f}%")
print(f"Invalid matrices: {invalid_count}")
print("\n")

# Determinant distribution with matrices
det_counts = Counter(canonical_strings.values())
print("Det distribution with sample matrices:")
print("=" * 50)

sort_dets = sorted(det_counts.items(), reverse=True)
for rank, (det, count) in enumerate(sort_dets, start=1):
    print(f"\nDeterminant: {det}, Count: {count}")
    
    # Print up to 2 example matrices for this determinant
    if rank <= 5:
        matrices_for_det = det_to_matrices[det]
        num_to_show = min(2, len(matrices_for_det))
        
        for i in range(num_to_show):
            print(f"Example matrix {i+1}:")
            print(matrices_for_det[i])
            if i < num_to_show - 1:
                print()
        
        if len(matrices_for_det) > num_to_show:
            print(f"... and {len(matrices_for_det) - num_to_show} more matrices with det = {det}")
    print("=" * 50)

In [13]:
import networkx as nx
import numpy as np

def equivalent_via_graph(A, B):
    A = np.array(A)
    B = np.array(B)
    n, m = A.shape
    if B.shape != (n, m):
        return False

    G1 = nx.Graph()
    G2 = nx.Graph()

    # Add bipartite nodes: rows and columns
    G1.add_nodes_from(range(n), bipartite=0)
    G1.add_nodes_from(range(n, n+m), bipartite=1)
    
    G2.add_nodes_from(range(n), bipartite=0)
    G2.add_nodes_from(range(n, n+m), bipartite=1)

    # Add edges for 1 entries
    for i in range(n):
        for j in range(m):
            if A[i, j]:
                G1.add_edge(i, n + j)
            if B[i, j]:
                G2.add_edge(i, n + j)

    # Check isomorphism
    return nx.is_isomorphic(G1, G2)

def decode_matrix(line):
    """decode a line of 0/1 string to a matrix"""
    try: 
        row_strs = line.strip().split(",")

        if len(row_strs) != N:
            return None
        
        matrix = []
        for row in row_strs:
            # validate row
            if len(row) != N:
                return None
            if not all(c in "01" for c in row):
                return None

            matrix.append([int(c) for c in row])

        matrix = np.array(matrix, dtype=int)

        if matrix.shape != (N, N):
            return None  
        return matrix
    
    except Exception:
        return None


In [15]:
import numpy as np
from collections import defaultdict, Counter

matrices = []
N = 16

file_path = "C:/Users/123li/Downloads/Project/gpu_run_output/dim16_run_9/1111/transformer-output-decoded.txt"
with open(file_path, encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

for line in lines:
    A = decode_matrix(line)
    if A is not None and A.shape == (N, N):
        matrices.append(A)

if matrices:
    print("Total matrices loaded for equivalence check =", len(matrices))

# step 1: group by determinant value
det_groups = defaultdict(list)
det_values = []
for A in matrices:
    det = round(abs(np.linalg.det(A)))
    det_groups[det].append(A)
    det_values.append(det)

det_counts = Counter(det_values)

# step 2: only focus on the top 5 determinant values and its count
top5 = sorted(det_counts.items(), key=lambda x: x[0], reverse=True)[:5]

print("Top 5 determinant groups:\n")
for i, (det, count) in enumerate(top5, start=1):
    print(f"{i}. det = {det}, count = {count}")

# step 3: within these top 5 groups, use graph isomorphism to find unique matrices
def unique_in_group(matrices):
    # input: list of matrices in the same determinant group
    # output: list of unique matrices under row/column permutations
    # based on equivalent_via_graph function
    unique_matrices = []

    for A in matrices:
        is_new = True

        for B in unique_matrices:
            if equivalent_via_graph(A, B):
                is_new = False
                break

        if is_new:
            unique_matrices.append(A)

    return unique_matrices

# step 4: get the number of unique matrices in each top 5 determinant group
# and print out some sample matrices
for det, count in top5:
    group_matrices = det_groups[det]

    print(f"\nDeterminant: {det}, Total Count: {count}")

    # if there is only one matrix in the group, no need to check equivalence
    if count == 1:
        print("Only one matrix in this group.")
        print("unique matrix:")
        print(group_matrices[0])
        continue

    unique_list = unique_in_group(group_matrices)
    unique_count = len(unique_list)
    print(f"Unique matrices under row/column permutations: {unique_count}")

    # print up to 2 sample unique matrices
    num_to_show = min(2, unique_count)
    for i in range(num_to_show):
        print(f"Sample unique matrix {i+1}:")
        print(unique_list[i])
        if i < num_to_show - 1:
            print()

Total matrices loaded for equivalence check = 1088
Top 5 determinant groups:

1. det = 327680, count = 10
2. det = 299008, count = 14
3. det = 270336, count = 18
4. det = 258560, count = 1
5. det = 257664, count = 9

Determinant: 327680, Total Count: 10
Unique matrices under row/column permutations: 1
Sample unique matrix 1:
[[1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0]
 [1 1 1 1 1 1 0 0 0 0 1 1 1 1 0 0]
 [0 0 1 1 1 1 0 1 1 0 0 1 1 0 1 1]
 [0 1 1 0 1 1 1 0 1 0 1 0 0 1 1 1]
 [1 1 0 1 0 1 1 1 0 0 0 0 1 1 1 1]
 [1 0 1 1 1 0 1 0 0 1 1 0 1 0 1 1]
 [1 1 1 0 1 0 0 1 0 1 0 1 0 1 1 1]
 [1 1 0 1 0 1 0 0 1 1 1 1 0 0 1 1]
 [1 1 1 0 0 0 1 1 1 0 1 1 1 0 1 0]
 [1 0 0 1 1 0 1 1 1 0 1 1 0 1 0 1]
 [0 0 1 1 0 1 1 1 0 1 1 1 0 1 1 0]
 [0 1 1 1 0 0 0 1 1 1 1 0 1 1 0 1]
 [0 1 0 1 1 0 1 0 1 1 0 1 1 1 1 0]
 [0 1 0 0 1 1 1 1 0 1 1 1 1 0 0 1]
 [1 0 0 0 1 1 0 1 1 1 1 0 1 1 1 0]
 [1 0 1 0 0 1 1 0 1 1 0 1 1 1 0 1]]

Determinant: 299008, Total Count: 14
Unique matrices under row/column permutations: 1
Sample unique matrix 1:
[

In [ ]:
# brute force pairwise comparison
unique_groups = []
matrices = [...]

file_path = "C:/Users/123li/Downloads/Project/gpu_run_output/dim16_run_9/1111/transformer-output-decoded.txt"
with open(file_path, encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

for line in lines:
    A = decode_matrix(line)
    if A is not None:
        matrices.append(A)

if matrices:
    print("Total matrices loaded for equivalence check =", len(matrices))

for i, A in enumerate(matrices):
    placed = False
    for group in unique_groups:
        B = group[0]               # compare with representative
        if equivalent_via_graph(A, B):
            group.append(A)
            placed = True
            break
    if not placed:
        unique_groups.append([A])  # new class

print("Total unique groups =", len(unique_groups))